# Notebook 2: K-Means Clustering — Two Features

**Dataset:** Survey of Consumer Finances (SCF) 2019  
**Method:** K-Means clustering using two features (Home Value & Mortgage Debt)

In this notebook we:
- Build a `wrangle` function for the TURNFEAR subset
- Construct a 2-feature matrix X
- Fit K-Means and extract cluster labels and centroids
- Evaluate with inertia and silhouette score
- Find the optimal K using elbow and silhouette curves
- Visualize the final clusters

## 0. Colab Setup

Run this cell first when working in Google Colab. It installs dependencies and mounts your Google Drive.

> **Before running:** Upload the `customer-segmentation/` folder to your Google Drive (or clone your GitHub repo).

In [ ]:
# ── Clone the GitHub repository ────────────────────────────────────────────
# Replace the URL below with your actual GitHub repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'

!git clone {REPO_URL}

# ── Set working directory to the project folder ────────────────────────────
import os

# Replace YOUR_REPO_NAME with the actual cloned folder name
os.chdir('/content/YOUR_REPO_NAME/customer-segmentation')

# ── Install required packages ──────────────────────────────────────────────
!pip install pandas numpy plotly scikit-learn scipy --quiet

print(f'Working directory: {os.getcwd()}')
print(f'Data file exists: {os.path.exists("data/SCF_2019.csv.gz")}')

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '${:,.2f}'.format)

print('Libraries loaded!')

## 2. Wrangle Function

In [ ]:
def wrangle(filepath):
    """
    Load and wrangle SCF 2019 data.
    
    Parameters
    ----------
    filepath : str
        Path to the compressed CSV file (SCF_2019.csv.gz)
    
    Returns
    -------
    pd.DataFrame
        Subset of households that were turned down for credit
        or feared being denied credit (TURNFEAR == 1).
        Uses implicate 1 only (every 5th row).
    """
    # Load data
    df = pd.read_csv(filepath, compression='gzip', index_col=0)
    
    # Keep implicate 1 (every 5th row)
    df = df.iloc[::5].reset_index(drop=True)
    
    # Subset: households turned down or fearing credit denial
    mask = df['TURNFEAR'] == 1
    df = df[mask].copy().reset_index(drop=True)
    
    return df


# Load the data
df = wrangle('data/SCF_2019.csv.gz')

print(f'Shape: {df.shape}')
print(f'Households with TURNFEAR == 1: {len(df):,}')
df.head()

## 3. Feature Selection: Two Features

We'll use **Home Value** (`HOUSES`) and **Mortgage Debt** (`MRTHEL`) as our two clustering features.

In [ ]:
# Feature matrix with two features
features = ['HOUSES', 'MRTHEL']

X = df[features].copy()

print(f'Feature matrix shape: {X.shape}')
print(f'\nFeature descriptions:')
print(f'  HOUSES  — Home/primary residence value')
print(f'  MRTHEL  — Mortgage debt on primary residence')
print()
X.describe()

## 4. Build Initial K-Means Model (K=3)

In [ ]:
# Build and fit K-Means model with K=3
model = KMeans(n_clusters=3, random_state=42, n_init=10)
model.fit(X)

print('K-Means model fitted successfully!')
print(f'Number of clusters: {model.n_clusters}')
print(f'Number of iterations: {model.n_iter_}')

## 5. Extract Cluster Labels

In [ ]:
# Extract cluster labels
labels = model.labels_

print('Cluster label distribution:')
label_counts = pd.Series(labels).value_counts().sort_index()
for cluster, count in label_counts.items():
    print(f'  Cluster {cluster}: {count:,} households ({count/len(labels)*100:.1f}%)')

# Add labels to X
X_labeled = X.copy()
X_labeled['Cluster'] = labels.astype(str)

## 6. Scatter Plot of Clusters

In [ ]:
# Filter extremes for visualization
X_plot = X_labeled[
    (X_labeled['HOUSES'] < X_labeled['HOUSES'].quantile(0.98)) &
    (X_labeled['MRTHEL'] < X_labeled['MRTHEL'].quantile(0.98))
].copy()

fig = px.scatter(
    X_plot,
    x='HOUSES',
    y='MRTHEL',
    color='Cluster',
    title='K-Means Clustering: Home Value vs. Mortgage Debt (K=3)',
    labels={'HOUSES': 'Home Value ($)', 'MRTHEL': 'Mortgage Debt ($)', 'Cluster': 'Cluster'},
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig.show()

## 7. Extract and Plot Centroids

In [ ]:
# Extract centroids
centroids = model.cluster_centers_
centroids_df = pd.DataFrame(centroids, columns=features)
centroids_df.index.name = 'Cluster'
centroids_df.index = [f'Cluster {i}' for i in range(len(centroids_df))]

print('Cluster Centroids:')
print(centroids_df.to_string())

# Plot clusters + centroids
fig = px.scatter(
    X_plot,
    x='HOUSES',
    y='MRTHEL',
    color='Cluster',
    title='K-Means Clusters with Centroids (K=3)',
    labels={'HOUSES': 'Home Value ($)', 'MRTHEL': 'Mortgage Debt ($)'},
    opacity=0.5,
    color_discrete_sequence=px.colors.qualitative.Set1
)

# Add centroids as larger markers
fig.add_scatter(
    x=centroids[:, 0],
    y=centroids[:, 1],
    mode='markers',
    marker=dict(size=18, symbol='x', color='black', line_width=3),
    name='Centroids'
)
fig.show()

## 8. Model Evaluation: Inertia and Silhouette Score

In [ ]:
# Calculate inertia
inertia = model.inertia_

# Calculate silhouette score
sil_score = silhouette_score(X, labels)

print(f'K-Means Model Evaluation (K=3):')
print(f'  Inertia:         {inertia:,.2f}')
print(f'  Silhouette Score: {sil_score:.4f}  (range: -1 to 1, higher is better)')

## 9. Finding Optimal K: Elbow and Silhouette Curves

In [ ]:
# Test K from 2 to 12
k_range = range(2, 13)
inertia_errors = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertia_errors.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X, km.labels_))
    print(f'K={k:2d} | Inertia: {km.inertia_:>18,.0f} | Silhouette: {silhouette_score(X, km.labels_):.4f}')

print('\nDone!')

## 10. Elbow Plot: Inertia vs. Number of Clusters

In [ ]:
fig = px.line(
    x=list(k_range),
    y=inertia_errors,
    markers=True,
    title='K-Means Model: Inertia vs Number of Clusters',
    labels={'x': 'Number of Clusters (K)', 'y': 'Inertia'}
)
fig.update_traces(line_color='#636EFA', marker_color='#EF553B', marker_size=8)
fig.update_layout(xaxis=dict(tickmode='linear', tick0=2, dtick=1))
fig.show()

## 11. Silhouette Score vs. Number of Clusters

In [ ]:
fig = px.line(
    x=list(k_range),
    y=silhouette_scores,
    markers=True,
    title='K-Means Model: Silhouette Score vs Number of Clusters',
    labels={'x': 'Number of Clusters (K)', 'y': 'Silhouette Score'}
)
fig.update_traces(line_color='#00CC96', marker_color='#EF553B', marker_size=8)
fig.update_layout(xaxis=dict(tickmode='linear', tick0=2, dtick=1))
fig.show()

best_k = list(k_range)[np.argmax(silhouette_scores)]
print(f'Best K by silhouette score: K={best_k} (score={max(silhouette_scores):.4f})')

## 12. Final Model

Based on the elbow and silhouette plots, choose the best K.

In [ ]:
# Build final model with the best K (adjust based on your plots)
BEST_K = best_k  # Change this based on your elbow/silhouette analysis

final_model = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
final_model.fit(X)

final_labels = final_model.labels_
final_inertia = final_model.inertia_
final_sil = silhouette_score(X, final_labels)

print(f'Final Model (K={BEST_K})')
print(f'  Inertia:          {final_inertia:,.2f}')
print(f'  Silhouette Score: {final_sil:.4f}')

label_counts = pd.Series(final_labels).value_counts().sort_index()
print(f'\nCluster sizes:')
for cluster, count in label_counts.items():
    print(f'  Cluster {cluster}: {count:,} ({count/len(final_labels)*100:.1f}%)')

## 13. Final Cluster Scatter Plot

In [ ]:
X_final = X.copy()
X_final['Cluster'] = final_labels.astype(str)

# Filter extremes
X_final_plot = X_final[
    (X_final['HOUSES'] < X_final['HOUSES'].quantile(0.98)) &
    (X_final['MRTHEL'] < X_final['MRTHEL'].quantile(0.98))
]

fig = px.scatter(
    X_final_plot,
    x='HOUSES',
    y='MRTHEL',
    color='Cluster',
    title=f'Final K-Means Clusters: Home Value vs. Mortgage Debt (K={BEST_K})',
    labels={'HOUSES': 'Home Value ($)', 'MRTHEL': 'Mortgage Debt ($)'},
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Plotly
)

# Add centroids
final_centroids = final_model.cluster_centers_
fig.add_scatter(
    x=final_centroids[:, 0],
    y=final_centroids[:, 1],
    mode='markers',
    marker=dict(size=18, symbol='x', color='black', line_width=3),
    name='Centroids'
)
fig.show()

## 14. Side-by-Side Bar Chart: Mean Features by Cluster

In [ ]:
# Compute mean of the two features per cluster
X_final_means = X.copy()
X_final_means['Cluster'] = final_labels
cluster_means = X_final_means.groupby('Cluster')[features].mean().reset_index()
cluster_means['Cluster'] = cluster_means['Cluster'].astype(str)

# Melt for grouped bar chart
cluster_means_melted = cluster_means.melt(
    id_vars='Cluster',
    value_vars=features,
    var_name='Feature',
    value_name='Mean Value'
)

fig = px.bar(
    cluster_means_melted,
    x='Cluster',
    y='Mean Value',
    color='Feature',
    barmode='group',
    title=f'Mean Feature Values by Cluster (K={BEST_K})',
    labels={'Cluster': 'Cluster', 'Mean Value': 'Mean Value ($)', 'Feature': 'Feature'},
    color_discrete_sequence=px.colors.qualitative.Plotly
)
fig.show()

print('Mean values by cluster:')
print(cluster_means.to_string(index=False))